In [ ]:
# Universal setup: auto-detects Colab vs local, mounts drive only if needed
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())
from config import mount_drive_if_colab, setup_directories, install_requirements

mount_drive_if_colab()

Mounted at /content/drive


In [ ]:
# Install requirements (safe to re-run; skip if already installed)
install_requirements()

# Imports

In [ ]:
import os
import pandas as pd
from unidecode import unidecode
from multiprocessing import Pool
import pickle
import warnings
import numpy as np

from tensorflow.keras.models import load_model
from keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings("ignore")

# Constants

In [ ]:
# Universal path setup — works in Colab AND local without any changes
dirs = setup_directories()

base_dir  = dirs["base"]
data_dir  = dirs["data"] + os.sep
model_dir = dirs["models"] + os.sep
output_dir = dirs["output"] + os.sep

print(f"Base directory:   {base_dir}")
print(f"Data directory:   {data_dir}")
print(f"Model directory:  {model_dir}")
print(f"Output directory: {output_dir}")

# Name of data file
dataname = "sample_data_race.csv"

# Check Data

In [ ]:
df=pd.read_csv(data_dir + dataname, encoding='utf-8')
print(len(df))
print(df.head())
del df

50
                  fullname    pz_whi    pz_bla    pz_his    pz_asi    pz_oth
0     JEFFREY TODD BUCKNER  0.847145  0.046353  0.072047  0.007551  0.026903
1       CAROL WILD BUCKNER  0.780187  0.127414  0.038097  0.028054  0.026247
2       JILL WINTERS HAGER  0.741789  0.132648  0.085556  0.022901  0.017106
3        NANCY WAGNER PIKE  0.388778  0.367163  0.161937  0.042781  0.039341
4  ROBERT GREGORY COCKROFT  0.525449  0.067754  0.047286  0.324763  0.034748


In [ ]:
#Set constants and variable names

colname = "fullname"
(pz_whi, pz_bla, pz_his, pz_asi, pz_oth) = ("pz_whi", "pz_bla", "pz_his", "pz_asi", "pz_oth") #variables containing proportions of respective groups (modify if required)

usegis = False # set to True for using gis with ethnicity models

modelfile = f'cnn_USA_{"meta" if usegis else "text"}.h5'

In [ ]:
def normalize(word):
    """Transliterate Unicode to ASCII."""
    if not isinstance(word, str):
        return ""
    return unidecode(word)

def clean_data():
    """Read raw data, clean names, and save intermediate CSV."""
    df = pd.read_csv(data_dir + dataname, encoding="utf-8")
    df[colname] = df[colname].replace(np.nan, "", regex=True)

    n_workers = min(4, os.cpu_count() or 2)
    with Pool(n_workers) as p:
        df[colname] = p.map(normalize, df[colname])

    df[colname] = df[colname].str.replace(r"[0-9]{2,}", "", regex=True)
    df[colname] = df[colname].str.replace(r"[^a-zA-Z]", " ", regex=True)
    df[colname] = df[colname].str.upper()
    df[colname] = df[colname].str.replace(r"\s+", " ", regex=True).str.strip()

    df.to_csv(output_dir + dataname, encoding="utf-8", index=False)
    print(f"Cleaned {len(df)} records → {output_dir + dataname}")

def load_data():
    """Load cleaned data, optionally with GIS prior probabilities."""
    data_test = pd.read_csv(output_dir + dataname)
    if usegis:
        default_prob = 1.0 / 5  # uniform prior; adjust based on state demographics
        return (
            data_test[colname],
            data_test[pz_whi].fillna(default_prob),
            data_test[pz_bla].fillna(default_prob),
            data_test[pz_his].fillna(default_prob),
            data_test[pz_asi].fillna(default_prob),
            data_test[pz_oth].fillna(default_prob),
        )
    else:
        return data_test[colname]

clean_data()

if usegis:
    x_test, pz_whi_test, pz_bla_test, pz_his_test, pz_asi_test, pz_oth_test = load_data()
else:
    x_test = load_data()

print(f"Loaded {len(x_test)} records for prediction.")

In [ ]:
# Load pre-trained CNN model, encoder, and tokenizer
model_path = model_dir + modelfile
assert os.path.isfile(model_path), f"Model not found: {model_path}\nPlace pre-trained .h5 models in {model_dir}"
model = load_model(model_path)
print(f"Loaded model: {modelfile}")

enc_path = model_dir + "nc_voter_encoding.pkl"
tok_path = model_dir + "nc_voter_tokenizer.pkl"
assert os.path.isfile(enc_path), f"Encoder not found: {enc_path}"
assert os.path.isfile(tok_path), f"Tokenizer not found: {tok_path}"

with open(enc_path, "rb") as f:
    encoder = pickle.load(f)

with open(tok_path, "rb") as f:
    (vocab_size, tokenizer, max_char) = pickle.load(f)

tagset = ["A", "B", "H", "O", "W"]

x = tokenizer.texts_to_sequences(x_test)
x = pad_sequences(x, maxlen=max_char, padding="post", truncating="post")

if usegis:
    x = [x, pz_whi_test, pz_bla_test, pz_his_test, pz_asi_test, pz_oth_test]

y_pred_prob = model.predict(x, batch_size=1024, verbose=1, use_multiprocessing=True, workers=-1)
y_pred = pd.Series(encoder.inverse_transform(y_pred_prob))

df = pd.concat([
    pd.read_csv(data_dir + dataname, encoding="utf-8"),
    pd.DataFrame({"predicted": y_pred}),
    pd.DataFrame(y_pred_prob, columns=tagset),
], axis=1)

out_path = output_dir + dataname
df.to_csv(out_path, index=False)
print(f"Predictions saved → {out_path}")
print(df.head())

1/1 [==============================] - 0s 246ms/step
